# 蒸留3塔 段プロファイル CSV 出力 (スライド作図用) — trial #201

**目的**: Dist1/Dist2/Dist3 の段ごとプロファイル (温度・液組成 x・気組成 y・圧力) を CSV 出力。
段は **塔頂(凝縮器)=1 → 塔底(リボイラ)=末尾**。

## 方針 (確定・実証済)
- **rigorous は精度不足で禁止**、**SM は段プロファイルを持たない** → 段プロファイルは **HYSYS のみ**。
- `HYSYS.Application` は単一 COM サーバで **3塔同時起動は不可**。さらに同一プロセスで
  HSC を開き直すと COM が不安定化し Dist3 で RPC 例外が出た。
  → **各塔を別プロセスで逐次実行**する (`tools/_run_dist_profiles.py <col>` を subprocess 起動)。
- 実行には Aspen HYSYS ライセンス (家PCは VPN で京大ライセンスサーバ到達) が必要。

## 段プロファイルの HYSYS COM アクセサ (probe で確定)
すべて `col.ColumnFlowsheet` 上。長さは **N+2** (凝縮器 + トレイN + リボイラ)。
- `TemperaturesValue` … 段温度 [°C]
- `PressuresValue` … 段圧力 [kPa]
- `LiquidMoleFractionsValue` … **成分×段** の液組成 (ncomp × (N+2))
- `VapourMoleFractionsValue` … **成分×段** の気組成
- 成分順は HYSYS fluid package 順 → 名前マッチ (`build_pdh_to_hysys_index`) で PDH キーへ対応。

## 設計点 (trial #201、推測なし)
`outputs/main_20260605_170938/best.json` + `top1_trial201.txt` の収束ストリーム実値。
各塔の条件・フィード組成は `tools/_run_dist_profiles.py` の `COLUMN_CONFIG` に固定。

## 注意 (HYSYS実行で判明)
- **Dist1 / Dist3 は HSC 側で Feed 温度が固定変数** (書込み拒否) のため温度書込みをスキップ。
  圧力・流量・組成・spec・フィード段は #201 値で設定。塔頂/塔底温度は #201 と一致を確認済。
- **Dist3 純度**: Draw Rate を recovery=0.5984 (= #201 塔頂C3H6/フィードC3H6) で与え、
  塔頂 C3H6 ≈ 99.46 wt% (mol 99.48%) に収束 (#201 SM は 99.497wt%、ほぼ一致)。
  厳密に 99.5wt% へ寄せたい場合は recovery を微調整 (下げると純度↑)。

## 出力先
まず `Z:\pdh_simulator\monitor\prof_dist{1,2,3}.csv`。確定後に末尾セルで `Y:\pdh_slide\figures\` へコピー。

In [ ]:
import os, sys, subprocess

PROJECT_ROOT = r"Z:\pdh_simulator"
PYEXE   = os.path.join(PROJECT_ROOT, r".venv\Scripts\python.exe")
SCRIPT  = os.path.join(PROJECT_ROOT, r"tools\_run_dist_profiles.py")
OUT_DIR = os.path.join(PROJECT_ROOT, "monitor")
FINAL_DIR = r"Y:\pdh_slide\figures"
OUT_CSV = {'dist1': 'prof_dist1.csv', 'dist2': 'prof_dist2.csv', 'dist3': 'prof_dist3.csv'}
print('python :', PYEXE)
print('script :', SCRIPT)

## 実行: 各塔を別プロセスで逐次 (COM 安定化)
下のセルは Dist1→Dist2→Dist3 を 1 塔ずつ別プロセスで起動する。各塔フレッシュな HYSYS を
立ち上げるため、同一カーネル内で連続実行しても COM が干渉しない。
(1塔だけ回したいときは `run_column('dist2')` のように個別実行も可。)

In [ ]:
def run_column(col: str):
    """tools/_run_dist_profiles.py <col> を別プロセスで実行し、出力を表示する。"""
    print(f"\n=========== {col} (別プロセス起動) ===========")
    p = subprocess.run([PYEXE, SCRIPT, col], cwd=PROJECT_ROOT,
                       capture_output=True, text=True, encoding='utf-8')
    print(p.stdout)
    if p.stderr.strip():
        print('--- stderr ---'); print(p.stderr[-2000:])
    print(f"=========== {col} exit={p.returncode} ===========")
    return p.returncode == 0

results = {c: run_column(c) for c in ('dist1', 'dist2', 'dist3')}
print('\n==== ALL ====', results)

## 生成 CSV の確認 (先頭・中間・末尾)

In [ ]:
import csv
for col, name in OUT_CSV.items():
    path = os.path.join(OUT_DIR, name)
    if not os.path.exists(path):
        print(f"[{col}] 未生成: {path}"); continue
    with open(path, encoding='utf-8') as f:
        rows = list(csv.reader(f))
    hdr, body = rows[0], rows[1:]
    print(f"\n### {name}  ({len(body)} 段)  列={hdr}")
    for j in (0, len(body)//2, len(body)-1):
        print('  ', body[j])

## 確定後: Y:\pdh_slide\figures へコピー

In [ ]:
import shutil
os.makedirs(FINAL_DIR, exist_ok=True)
for name in OUT_CSV.values():
    src = os.path.join(OUT_DIR, name)
    if os.path.exists(src):
        dst = os.path.join(FINAL_DIR, name)
        shutil.copyfile(src, dst); print(f"copied: {src} -> {dst}")
    else:
        print(f"skip (未生成): {src}")